# Comprehensive Agent AI with LangGraph: RAG Framework Tutorial

## 🚀 Welcome to the Complete Agent AI Code Lab!

This comprehensive tutorial will guide you through building a sophisticated **Retrieval-Augmented Generation (RAG)** agent using **LangGraph**. You'll learn to create an intelligent system that can retrieve information from documents, generate responses, and evaluate answer quality.

### 🎯 What You'll Build:
- **Retrieval Node**: Index and search through documents using vector embeddings
- **Generation Node**: Generate contextual answers using retrieved information  
- **Grader Node**: Evaluate and score the quality of generated responses
- **Complete RAG Pipeline**: Integrate all components into a single LangGraph workflow

### 🎓 Learning Objectives:
By the end of this lab, you will understand:
- Agent AI fundamentals and LangGraph architecture
- State and context management in AI systems
- Vector store implementation and document retrieval
- RAG framework design and implementation
- Response evaluation and quality grading
- Real-world applications and use cases

Let's get started! 🚀

## 📚 Table of Contents

1. [Introduction to Agent AI and LangGraph](#introduction)
2. [Installation and Setup](#setup)
3. [Understanding State and Memory](#state-management)
4. [Building the Document Retrieval System](#retrieval-node)
5. [Creating the AI Response Generator](#generation-node)
6. [Building the Quality Evaluator](#grader-node)
7. [Putting It All Together - Complete RAG System](#rag-pipeline)
8. [Testing with Real Examples](#examples)
9. [Troubleshooting Common Issues](#troubleshooting)

---

## 1. Introduction to Agent AI and LangGraph {#introduction}

### 🤖 What is Agent AI?

**Agent AI** refers to autonomous AI systems that can perceive their environment, make decisions, and take actions to achieve specific goals. Unlike traditional AI models that simply respond to prompts, agents can:

- **Plan and Execute**: Break down complex tasks into manageable steps
- **Use Tools**: Interact with external systems, APIs, and databases
- **Learn and Adapt**: Improve performance based on feedback and experience
- **Maintain Context**: Remember previous interactions and build upon them

### 🌐 What is LangGraph?

**LangGraph** is a powerful framework for building stateful, multi-actor applications with Large Language Models (LLMs). It extends LangChain with:

- **Graph-based Architecture**: Define workflows as directed graphs
- **State Management**: Maintain and update shared state across nodes
- **Conditional Logic**: Dynamic routing based on conditions
- **Parallel Execution**: Run multiple nodes simultaneously
- **Human-in-the-Loop**: Integrate human feedback and oversight

### 🏢 Real-World Applications

#### Customer Support Automation
- **Document Retrieval**: Find relevant policy documents and FAQs
- **Response Generation**: Create personalized, contextual responses
- **Quality Assurance**: Grade responses for accuracy and helpfulness

#### Research and Analysis
- **Literature Review**: Search through academic papers and reports
- **Evidence Synthesis**: Combine information from multiple sources
- **Fact Checking**: Verify claims against authoritative sources

#### Content Creation
- **Research Phase**: Gather relevant information and sources
- **Writing Phase**: Generate content based on retrieved information
- **Review Phase**: Evaluate content quality and accuracy

#### Legal and Compliance
- **Case Law Research**: Find relevant precedents and regulations
- **Document Analysis**: Extract key information from legal documents
- **Compliance Checking**: Verify adherence to regulatory requirements

### 🔄 The RAG Framework Advantage

**Retrieval-Augmented Generation (RAG)** combines the best of both worlds:
- **Factual Accuracy**: Grounds responses in real, up-to-date information
- **Contextual Relevance**: Provides specific, targeted answers to user queries
- **Source Attribution**: Shows where information comes from
- **Scalability**: Can work with vast document collections

## 2. Installation and Setup {#setup}

### 📋 Prerequisites Checklist

Before we begin, ensure you have:
- ✅ **Python 3.8+** (Check with `python --version`)
- ✅ **Jupyter Notebook** or **JupyterLab** 
- ✅ **OpenAI API Key** (Get one from [OpenAI Platform](https://platform.openai.com/))
- ✅ **Stable Internet Connection** for package downloads

### 🌍 Setting up Virtual Environment (Recommended)

```bash
# Create a new virtual environment
python -m venv rag_agent_env

# Activate the environment
# Windows:
rag_agent_env\Scripts\activate
# macOS/Linux:
source rag_agent_env/bin/activate

# Install Jupyter if not already available
pip install jupyter notebook ipywidgets
```

### 📦 Required Libraries Overview

We'll be installing several key libraries:
- **langgraph**: Core framework for building our agent
- **langchain**: Foundation for LLM interactions
- **openai**: OpenAI API integration
- **faiss-cpu**: Vector store for document indexing
- **sentence-transformers**: Text embeddings
- **pypdf**: PDF document processing
- **tiktoken**: Token counting for OpenAI models

In [ ]:
# Install all required packages for RAG Agent
print("🔧 Installing packages for RAG Agent with LangGraph...")

# %pip install --quiet pydantic
# %pip install --quiet langgraph langchain langchain-openai openai
# %pip install --quiet langchain-community langchain-text-splitters
# %pip install --quiet faiss-cpu sentence-transformers
# %pip install --quiet pypdf tiktoken python-dotenv
# %pip install --quiet numpy pandas matplotlib

print("✅ All packages installed successfully!")

In [ ]:
# Import all required libraries
import sys
from typing import TypedDict, List, Dict, Any

# LangGraph and LangChain imports
from langgraph.graph import StateGraph, END
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings

# Additional utilities
import numpy as np
import tiktoken

print("✅ All libraries imported successfully!")
print("🔍 Available components:")
print("  - StateGraph for workflow management")
print("  - OpenAI models for LLM and embeddings")
print("  - FAISS for vector storage")
print("  - Document processing utilities")
print("  - Evaluation and scoring tools")
print("Note: Using OpenAI embeddings instead of sentence_transformers to avoid dependency conflicts")

### Azure OpenAI Chat Model Integration

This example demonstrates how to use the `AzureChatOpenAI` class from the `langchain_openai` package to interact with Azure OpenAI's GPT models.

#### Key Steps:
1. **Setup**: Import necessary libraries and retrieve credentials for the Azure OpenAI API.
2. **Credentials**: Ensure the Azure API key and endpoint are correctly set.
3. **Error Handling**: Check if the API key and endpoint are missing, and raise errors if needed.
4. **Chat Model**: Initialize the `AzureChatOpenAI` model by providing required credentials like API key, endpoint, model type, and API version.

#### Notes:
- Ensure that Azure OpenAI is set up and environment variables for the API key and endpoint are configured.


In [ ]:
sys.path.insert(1, '../')

In [ ]:
import init_creds as creds

AZURE_OPENAI_KEY = creds.get_api_key()
AZURE_OPENAI_ENDPOINT = creds.get_endpoint()
AZURE_OPENAI_API_VERSION = "2025-04-01-preview"

if not AZURE_OPENAI_KEY:
    raise ValueError("No AZURE_OPENAI_KEY set for Azure OpenAI API")
if not AZURE_OPENAI_ENDPOINT:
    raise ValueError("No AZURE_OPENAI_ENDPOINT set for Azure OpenAI API")

llm = AzureChatOpenAI(
    model="gpt-4o-mini",
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_KEY,
    api_version=AZURE_OPENAI_API_VERSION
)

embeddings = AzureOpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=AZURE_OPENAI_KEY,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_version=AZURE_OPENAI_API_VERSION
    )

print("✅ LLM and Embeddings initialized successfully!")
print(f"📝 Language Model: {llm.model_name}")
print(f"🔍 Embeddings Model: text-embedding-3-small")

## 3. Understanding State and Memory {#state-management}

### 🧠 What is "State" in AI Systems?

Think of **state** like a shared notebook that our AI agent uses to remember things as it works. Just like when you're solving a math problem, you write down intermediate steps - our AI agent needs to remember:

- **What the user asked** - The original question
- **What documents it found** - Relevant information from its knowledge base
- **What answer it created** - The response it generated
- **How good the answer was** - A quality score

### 📝 Two Types of Memory

#### Short-term Memory (Like your working memory)
- **Current Conversation**: What we're talking about right now
- **Found Documents**: Information just retrieved for this question
- **Work in Progress**: Steps the AI is currently doing

#### Long-term Memory (Like a library)
- **Knowledge Base**: All the documents and information stored permanently
- **Learned Patterns**: What types of questions work best
- **Performance History**: How well the system has been working

### 🏗️ Creating Our AI Agent's "Notebook"

Let's create a structure to track everything our AI agent needs to remember:

In [ ]:
# Define our AI agent's "memory structure"
# Think of this like creating a form with different sections to fill out

class RAGAgentState(TypedDict):
    """
    This is like a digital form that our AI agent fills out as it works.
    Each piece of information gets stored in a specific field.
    """
    
    # 📝 WHAT THE USER WANTS TO KNOW
    user_query: str                    # The question the user asked
    messages: List[BaseMessage]        # Chat history (like a conversation log)
    
    # 🔍 WHAT DOCUMENTS WE FOUND
    retrieved_documents: List[Document] # The relevant documents we found
    retrieval_scores: List[float]      # How relevant each document is (0-1 scale)
    retrieval_query: str               # The search terms we used
    
    # 💭 WHAT ANSWER WE CREATED
    generated_response: str            # The final answer we gave to the user
    context_used: str                  # The information we used to create the answer
    generation_prompt: str             # The instructions we gave to the AI
    
    # 📊 HOW GOOD OUR ANSWER WAS
    relevance_score: float             # Does it answer the question? (0-10)
    accuracy_score: float              # Is the information correct? (0-10)
    completeness_score: float          # Is it a complete answer? (0-10)
    overall_grade: str                 # Letter grade like school (A, B, C, D, F)
    grading_explanation: str           # Why did it get this grade?
    
    # 🔧 SYSTEM INFORMATION (for debugging)
    step_count: int                    # How many steps did we take?
    processing_time: float             # How long did it take?
    tokens_used: int                   # How much "AI energy" did we use?
    error_messages: List[str]          # Any problems that occurred

# Helper function: Create a blank "form" for our AI agent to fill out
def create_initial_state(user_query: str) -> RAGAgentState:
    """
    This creates a fresh, empty form for our AI agent.
    It's like giving the agent a blank worksheet to fill out as it processes the question.
    """
    return RAGAgentState(
        # Start with the user's question
        user_query=user_query,
        messages=[],                    # No conversation history yet
        
        # No documents found yet
        retrieved_documents=[],
        retrieval_scores=[],
        retrieval_query="",
        
        # No answer created yet
        generated_response="",
        context_used="",
        generation_prompt="",
        
        # No grades assigned yet
        relevance_score=0.0,
        accuracy_score=0.0,
        completeness_score=0.0,
        overall_grade="",
        grading_explanation="",
        
        # Starting counters
        step_count=0,
        processing_time=0.0,
        tokens_used=0,
        error_messages=[]
    )

print("✅ RAG Agent State Schema defined!")
print("📊 State includes:")
print("  - Query processing and conversation history")
print("  - Document retrieval and scoring")
print("  - Response generation and context")
print("  - Quality grading and evaluation")
print("  - System performance tracking")

## 4. Building the Document Retrieval System {#retrieval-node}

### 🔍 What Does Our Document Finder Do?

Imagine you're in a huge library and need to find books related to your question. Our **Document Retrieval System** works like a super-smart librarian that:

1. **Organizes the Library**: Takes all our documents and creates a searchable catalog
2. **Understands Your Question**: Figures out what you're really asking for
3. **Finds Related Information**: Searches through everything to find the most relevant content
4. **Ranks by Relevance**: Puts the most helpful documents first

### 📚 How Does "Smart Search" Work?

Instead of just looking for exact word matches (like Ctrl+F), our system uses **semantic search** - it understands meaning:

- **Break Down Documents**: Split large documents into smaller, digestible chunks
- **Create "Fingerprints"**: Convert each chunk into a unique mathematical signature
- **Build a Searchable Index**: Store all these signatures in a way we can search quickly
- **Compare Meanings**: When you ask a question, we find chunks with similar meanings
- **Return Best Matches**: Give you the most relevant information

### 🏗️ Let's Build Our Smart Library System

In [ ]:
# Create sample documents for our knowledge base
# These are like the "books" in our AI library that it can search through

sample_documents = [
    """
    DOCUMENT 1: What is Artificial Intelligence?
    
    Artificial Intelligence (AI) is like teaching computers to think and learn like humans. 
    It's a field of computer science that creates smart machines that can do tasks that 
    usually need human intelligence, such as:
    - Learning from experience
    - Solving problems
    - Understanding language
    - Recognizing images
    
    There are three main types of AI:
    1. Narrow AI (Weak AI): Good at one specific task (like playing chess or recognizing faces)
    2. General AI (Strong AI): Could think like a human across many different areas (doesn't exist yet)
    3. Superintelligence: Smarter than humans in every way (still science fiction)
    
    Today's AI is used in: voice assistants (Siri, Alexa), movie recommendations (Netflix), 
    self-driving cars, medical diagnosis, and chatbots.
    """,
    
    """
    DOCUMENT 2: What is Machine Learning?
    
    Machine Learning (ML) is like teaching a computer to learn patterns by showing it lots of examples,
    instead of programming every single rule. It's a part of AI that helps computers make predictions
    or decisions based on data they've seen before.
    
    Think of it like this: Instead of telling a computer "if you see a picture with four legs and a tail, 
    it's probably a dog," we show it thousands of dog pictures and let it figure out the pattern.
    
    Three main types of Machine Learning:
    1. Supervised Learning: Learning with a teacher (showing correct answers during training)
       - Like flashcards: show the computer pictures labeled "dog" or "cat"
    2. Unsupervised Learning: Learning without a teacher (finding patterns on their own)
       - Like sorting mixed buttons by color without being told what colors exist
    3. Reinforcement Learning: Learning through trial and error (getting rewards/penalties)
       - Like learning to play a video game by trying different moves and seeing what works
    
    Common examples: Email spam detection, movie recommendations, voice recognition, price predictions.
    """,
    
    """
    DOCUMENT 3: What are Large Language Models (LLMs)?
    
    Large Language Models (LLMs) are like super-smart autocomplete systems that have read 
    almost everything on the internet. They can understand and write text that sounds very human-like.
    
    Popular examples: ChatGPT, GPT-4, Google's Bard, Claude (that's me!), and many others.
    
    What makes them special:
    - They've been "trained" by reading billions of web pages, books, and articles
    - They can write stories, answer questions, translate languages, and even write code
    - They use a special brain-like architecture called "transformers" (like how our brains have neurons)
    - They need massive computer power to run (think thousands of powerful computers working together)
    
    What can they do?
    - Have conversations (like chatbots)
    - Write essays, stories, and articles
    - Translate between languages
    - Summarize long documents
    - Help with coding
    - Answer questions about almost any topic
    
    Think of them as incredibly well-read assistants who can help with writing and thinking tasks!
    """,
    
    """
    DOCUMENT 4: What is RAG (Retrieval-Augmented Generation)?
    
    RAG is like giving an AI assistant access to a library when answering questions. Instead of just 
    relying on what it learned during training, it can look up current, specific information to give better answers.
    
    Here's how RAG works (like a helpful librarian):
    1. RETRIEVAL: "Let me search for information about your question"
       - The AI searches through its knowledge base to find relevant documents
    2. AUGMENTATION: "I found some helpful information, let me read it"
       - It takes the most relevant documents and adds them to your question
    3. GENERATION: "Based on what I found, here's your answer"
       - It creates a response using both your question AND the information it found
    
    Why is RAG awesome?
    - More accurate answers: It uses real, specific information rather than just memory
    - Up-to-date information: Can access recent documents and data
    - Less "making stuff up": Reduces AI hallucinations by grounding answers in real sources
    - Shows its work: Can tell you where the information came from
    - Cost-effective: Cheaper than training huge AI models with more data
    
    Think of it as the difference between answering from memory vs. looking it up in a book!
    """
]

# Process documents and create our searchable library
def create_document_index(documents: List[str]) -> FAISS:
    """
    This function creates our "smart library" from a list of documents.
    
    Think of it like:
    1. Taking a big book and cutting it into smaller, manageable pages
    2. Creating a "fingerprint" for each page so we can find it quickly
    3. Building a searchable catalog of all the pages
    
    Args:
        documents: List of text documents (our "books")
        
    Returns:
        A searchable library (FAISS vector store) ready to answer questions
    """
    print("📚 Creating our smart library from documents...")
    
    # STEP 1: Split big documents into smaller chunks (like book pages)
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,        # Each "page" will have about 500 characters
        chunk_overlap=50,      # Pages overlap by 50 characters (to avoid cutting sentences)
        separators=["\n\n", "\n", ".", "!", "?", ",", " ", ""]  # Where to split text naturally
    )
    
    # STEP 2: Create our library "catalog cards" for each page
    docs = []
    for i, doc_text in enumerate(documents):
        # Split each document into smaller chunks
        chunks = text_splitter.split_text(doc_text)
        for j, chunk in enumerate(chunks):
            # Create a "catalog card" for each chunk
            docs.append(Document(
                page_content=chunk,  # The actual text content
                metadata={"source": f"document_{i}", "chunk": j}  # Info about where it came from
            ))
    
    print(f"📄 Created {len(docs)} searchable pages from our documents")
    
    # STEP 3: Build the searchable library index
    try:
        print("🔨 Building the searchable index (this creates 'fingerprints' for each page)...")
        vector_store = FAISS.from_documents(docs, embeddings)
        print("✅ Smart library created successfully!")
        print("🎯 Your AI can now search through all documents to answer questions!")
        return vector_store
    except Exception as e:
        print(f"❌ Error creating the library: {e}")
        print("💡 Try checking your internet connection and API key")
        return None

# Create our knowledge base
vector_store = create_document_index(sample_documents)

if vector_store:
    print(f"🎯 Knowledge base ready with {vector_store.index.ntotal} vectors")

In [ ]:
# Implement the Retrieval Node
def retrieval_node(state: RAGAgentState) -> RAGAgentState:
    """
    Retrieval node that finds relevant documents for the user query.
    
    Process:
    1. Optimize the user query for search
    2. Perform similarity search in vector store  
    3. Rank and filter results
    4. Update state with retrieved documents
    """
    print(f"🔍 RETRIEVAL NODE: Processing query: '{state['user_query']}'")
    
    try:
        # Optimize query for better retrieval
        query = state["user_query"].strip()
        if not query:
            return {
                **state,
                "error_messages": state["error_messages"] + ["Empty query provided"],
                "step_count": state["step_count"] + 1
            }
        
        # Perform similarity search
        search_results = vector_store.similarity_search_with_score(
            query=query,
            k=4  # Retrieve top 4 most relevant documents
        )
        
        # Extract documents and scores
        retrieved_docs = [doc for doc, score in search_results]
        retrieval_scores = [float(score) for doc, score in search_results]
        
        print(f"📋 Retrieved {len(retrieved_docs)} documents")
        for i, (doc, score) in enumerate(search_results):
            print(f"  {i+1}. Score: {score:.4f} - {doc.page_content[:100]}...")
        
        # Update state
        updated_state = {
            **state,
            "retrieved_documents": retrieved_docs,
            "retrieval_scores": retrieval_scores,
            "retrieval_query": query,
            "step_count": state["step_count"] + 1
        }
        
        return updated_state
        
    except Exception as e:
        error_msg = f"Retrieval error: {str(e)}"
        print(f"❌ {error_msg}")
        return {
            **state,
            "error_messages": state["error_messages"] + [error_msg],
            "step_count": state["step_count"] + 1
        }

# Test the retrieval node
print("🧪 Testing Retrieval Node...")
test_state = create_initial_state("What is machine learning and how does it work?")
result_state = retrieval_node(test_state)

print(f"\n✅ Retrieval completed!")
print(f"📊 Retrieved {len(result_state['retrieved_documents'])} documents")
print(f"🎯 Average relevance score: {np.mean(result_state['retrieval_scores']):.4f}")

## 5. Creating the AI Response Generator {#generation-node}

### 🎯 What Does Our Answer Generator Do?

This is where the magic happens! Our **AI Response Generator** is like a smart writer that:

1. **Reads the Research**: Takes all the documents we found and reads through them
2. **Organizes the Information**: Puts the most important parts together in a logical way
3. **Writes a Custom Answer**: Creates a response specifically for the user's question
4. **Checks the Work**: Makes sure the answer makes sense and is well-formatted

### 🔄 How Does It Create Answers?

Think of it like writing a research paper, but super fast:

```
Found Documents → Read & Organize → Write Instructions → AI Creates Answer → Final Review
```

### 💡 Why This Approach is So Powerful

- **Based on Real Information**: Uses actual documents instead of just guessing
- **Shows Its Sources**: Can tell you where the information came from
- **Less Likely to Make Mistakes**: Grounded in real facts rather than AI "imagination"
- **Always Up-to-Date**: Can use the newest information you give it
- **Customized Answers**: Tailors the response specifically to what you asked

### 🛠️ Let's Build Our Smart Answer Writer

In [ ]:
def generation_node(state: RAGAgentState) -> RAGAgentState:
    """
    Generation node that creates responses using retrieved context and LLM.
    
    Process:
    1. Format retrieved documents into context
    2. Construct RAG prompt with query and context
    3. Call LLM to generate response
    4. Process and clean the response
    """
    print(f"🎯 GENERATION NODE: Creating response for: '{state['user_query']}'")
    
    try:
        # Check if we have retrieved documents
        if not state["retrieved_documents"]:
            return {
                **state,
                "generated_response": "I don't have enough information to answer your question.",
                "context_used": "",
                "generation_prompt": "",
                "error_messages": state["error_messages"] + ["No documents retrieved"],
                "step_count": state["step_count"] + 1
            }
        
        # Format context from retrieved documents
        context_parts = []
        for i, doc in enumerate(state["retrieved_documents"]):
            score = state["retrieval_scores"][i] if i < len(state["retrieval_scores"]) else 0.0
            context_parts.append(f"Document {i+1} (Relevance: {score:.3f}):\n{doc.page_content}\n")
        
        context = "\n".join(context_parts)
        
        # Construct the RAG prompt
        system_prompt = """You are a helpful AI assistant that answers questions based on the provided context.

INSTRUCTIONS:
1. Use ONLY the information provided in the context to answer the question
2. If the context doesn't contain enough information, say so clearly
3. Provide specific, accurate, and helpful responses
4. Cite relevant parts of the context when possible
5. Be concise but comprehensive
6. If asked about something not in the context, acknowledge the limitation

Remember: Your goal is to provide accurate, helpful answers grounded in the provided information."""

        user_prompt = f"""Context Information:
{context}

Question: {state['user_query']}

Please provide a comprehensive answer based on the context provided above."""

        # Create messages for LLM
        messages = [
            SystemMessage(content=system_prompt),
            HumanMessage(content=user_prompt)
        ]
        
        # Store the full prompt for debugging/grading
        full_prompt = f"System: {system_prompt}\n\nUser: {user_prompt}"
        
        print("🤖 Calling LLM for response generation...")
        
        # Call LLM
        response = llm.invoke(messages)
        generated_response = response.content.strip()
        
        print(f"✅ Response generated: {len(generated_response)} characters")
        
        # Count tokens used (approximate)
        try:
            encoding = tiktoken.encoding_for_model("gpt-3.5-turbo")
            tokens_used = len(encoding.encode(full_prompt + generated_response))
        except:
            tokens_used = len(full_prompt.split()) + len(generated_response.split())
        
        # Update state
        updated_state = {
            **state,
            "generated_response": generated_response,
            "context_used": context,
            "generation_prompt": full_prompt,
            "tokens_used": state["tokens_used"] + tokens_used,
            "step_count": state["step_count"] + 1
        }
        
        return updated_state
        
    except Exception as e:
        error_msg = f"Generation error: {str(e)}"
        print(f"❌ {error_msg}")
        return {
            **state,
            "generated_response": "I apologize, but I encountered an error while generating a response.",
            "error_messages": state["error_messages"] + [error_msg],
            "step_count": state["step_count"] + 1
        }

# Test the generation node with our previous retrieval result
print("🧪 Testing Generation Node...")
generation_result = generation_node(result_state)

print(f"\n📝 Generated Response:")
print(f"{generation_result['generated_response']}")
print(f"\n📊 Tokens used: {generation_result['tokens_used']}")
print(f"🔢 Total steps: {generation_result['step_count']}")

## 6. Implementing the Grader Node {#grader-node}

### 🎓 Understanding the Grader Node

The **Grader Node** provides crucial quality assessment for our RAG system. It evaluates responses across multiple dimensions:

1. **Relevance**: How well does the response address the user's question?
2. **Accuracy**: Is the information factually correct based on the context?
3. **Completeness**: Does the response fully answer the question?
4. **Quality**: Is the response well-structured and clear?

### 📊 Evaluation Metrics

Our grading system uses multiple criteria:

- **Relevance Score (0-10)**: Measures how directly the response answers the question
- **Accuracy Score (0-10)**: Evaluates factual correctness against source documents
- **Completeness Score (0-10)**: Assesses whether all parts of the question are addressed
- **Overall Grade (A-F)**: Final letter grade based on combined scores

### 🔍 Why Grading Matters

- **Quality Assurance**: Ensures responses meet minimum quality standards
- **Continuous Improvement**: Identifies areas for system optimization
- **User Trust**: Provides transparency about response reliability
- **Feedback Loop**: Enables iterative improvements to the RAG pipeline

### ⚙️ Implementing the Grader Node

In [ ]:
def grader_node(state: RAGAgentState) -> RAGAgentState:
    """
    Grader node that evaluates the quality of generated responses.
    
    Evaluates responses on:
    1. Relevance - How well does it answer the question?
    2. Accuracy - Is the information correct based on context?
    3. Completeness - Does it fully address the question?
    
    Returns updated state with scores and grades.
    """
    print(f"🎓 GRADER NODE: Evaluating response quality...")
    
    try:
        # Check if we have a response to grade
        if not state["generated_response"]:
            return {
                **state,
                "error_messages": state["error_messages"] + ["No response to grade"],
                "step_count": state["step_count"] + 1
            }
        
        # Prepare grading prompt
        grading_prompt = f"""
You are an expert evaluator assessing the quality of AI-generated responses in a RAG (Retrieval-Augmented Generation) system.

ORIGINAL QUESTION: {state['user_query']}

RETRIEVED CONTEXT: {state['context_used']}

GENERATED RESPONSE: {state['generated_response']}

Please evaluate this response on the following criteria (score 0-10 for each):

1. RELEVANCE (0-10): How well does the response directly address the original question?
   - 10: Perfectly addresses all aspects of the question
   - 5: Partially addresses the question
   - 0: Completely irrelevant to the question

2. ACCURACY (0-10): How factually correct is the response based on the provided context?
   - 10: All information is accurate and supported by context
   - 5: Mostly accurate with minor issues
   - 0: Contains significant factual errors

3. COMPLETENESS (0-10): How thoroughly does the response answer the question?
   - 10: Comprehensive answer covering all aspects
   - 5: Adequate answer but missing some details
   - 0: Incomplete or superficial answer

Provide your evaluation in this exact format:
RELEVANCE_SCORE: [score]
ACCURACY_SCORE: [score]  
COMPLETENESS_SCORE: [score]
EXPLANATION: [2-3 sentences explaining the scores]
"""

        # Call LLM for grading
        grading_messages = [
            SystemMessage(content="You are an expert evaluator for AI responses. Provide objective, consistent scoring."),
            HumanMessage(content=grading_prompt)
        ]
        
        print("🤖 Calling LLM for response evaluation...")
        grading_response = llm.invoke(grading_messages)
        grading_text = grading_response.content
        
        # Parse grading results
        relevance_score = 5.0  # Default scores
        accuracy_score = 5.0
        completeness_score = 5.0
        explanation = "Grading completed."
        
        # Extract scores from LLM response
        lines = grading_text.split('\n')
        for line in lines:
            line = line.strip()
            if line.startswith('RELEVANCE_SCORE:'):
                try:
                    relevance_score = float(line.split(':')[1].strip())
                except:
                    pass
            elif line.startswith('ACCURACY_SCORE:'):
                try:
                    accuracy_score = float(line.split(':')[1].strip())
                except:
                    pass
            elif line.startswith('COMPLETENESS_SCORE:'):
                try:
                    completeness_score = float(line.split(':')[1].strip())
                except:
                    pass
            elif line.startswith('EXPLANATION:'):
                explanation = line.split(':', 1)[1].strip()
        
        # Calculate overall grade
        average_score = (relevance_score + accuracy_score + completeness_score) / 3
        
        if average_score >= 9.0:
            overall_grade = "A"
        elif average_score >= 8.0:
            overall_grade = "B"
        elif average_score >= 7.0:
            overall_grade = "C"
        elif average_score >= 6.0:
            overall_grade = "D"
        else:
            overall_grade = "F"
        
        print(f"📊 Grading Results:")
        print(f"   Relevance: {relevance_score:.1f}/10")
        print(f"   Accuracy: {accuracy_score:.1f}/10")
        print(f"   Completeness: {completeness_score:.1f}/10")
        print(f"   Overall Grade: {overall_grade} (Avg: {average_score:.1f})")
        
        # Update state with grading results
        updated_state = {
            **state,
            "relevance_score": relevance_score,
            "accuracy_score": accuracy_score,
            "completeness_score": completeness_score,
            "overall_grade": overall_grade,
            "grading_explanation": explanation,
            "step_count": state["step_count"] + 1
        }
        
        return updated_state
        
    except Exception as e:
        error_msg = f"Grading error: {str(e)}"
        print(f"❌ {error_msg}")
        return {
            **state,
            "error_messages": state["error_messages"] + [error_msg],
            "step_count": state["step_count"] + 1
        }

# Test the grader node
print("🧪 Testing Grader Node...")
grading_result = grader_node(generation_result)

print(f"\n🎯 Final Evaluation:")
print(f"Question: {grading_result['user_query']}")
print(f"Response: {grading_result['generated_response'][:200]}...")
print(f"Grade: {grading_result['overall_grade']}")
print(f"Scores: R={grading_result['relevance_score']:.1f}, A={grading_result['accuracy_score']:.1f}, C={grading_result['completeness_score']:.1f}")
print(f"Explanation: {grading_result['grading_explanation']}")

## 7. Complete RAG Pipeline Integration {#rag-pipeline}

### 🔗 Building the Complete Workflow

Now we'll integrate all our nodes into a single, cohesive LangGraph workflow:

```
User Query → Retrieval Node → Generation Node → Grader Node → Final Response
```

### 🏗️ LangGraph Architecture

Our RAG system uses **StateGraph** to manage the flow of information:

1. **State Management**: Shared state flows through all nodes
2. **Sequential Processing**: Each node processes and updates the state
3. **Error Handling**: Robust error handling at each step
4. **Monitoring**: Track performance and quality metrics

### 🚀 Complete RAG Agent Implementation

In [ ]:
# Create the complete RAG workflow
def create_rag_workflow() -> StateGraph:
    """
    Create a complete RAG workflow with Retrieval, Generation, and Grading nodes.
    
    Returns:
        Compiled StateGraph ready for execution
    """
    print("🏗️ Building complete RAG workflow...")
    
    # Initialize the workflow
    workflow = StateGraph(RAGAgentState)
    
    # Add all nodes
    workflow.add_node("retrieval", retrieval_node)
    workflow.add_node("generation", generation_node)
    workflow.add_node("grader", grader_node)
    
    # Define the flow: Retrieval → Generation → Grader → END
    workflow.set_entry_point("retrieval")
    workflow.add_edge("retrieval", "generation")
    workflow.add_edge("generation", "grader")
    workflow.add_edge("grader", END)
    
    # Compile the workflow
    rag_agent = workflow.compile()
    
    print("✅ RAG workflow created successfully!")
    print("🔄 Flow: Retrieval → Generation → Grader → END")
    
    return rag_agent

# Create the RAG agent
rag_agent = create_rag_workflow()

# Helper function for easy interaction
def ask_rag_agent(question: str) -> Dict[str, Any]:
    """
    Ask a question to the RAG agent and get a complete response with evaluation.
    
    Args:
        question: User's question
        
    Returns:
        Dictionary with response, grades, and metadata
    """
    print(f"🤔 Question: {question}")
    print("=" * 80)
    
    # Create initial state
    initial_state = create_initial_state(question)
    
    # Run the complete RAG pipeline
    import time
    start_time = time.time()
    
    final_state = rag_agent.invoke(initial_state)
    
    end_time = time.time()
    processing_time = end_time - start_time
    
    # Update processing time
    final_state["processing_time"] = processing_time
    
    # Display results
    print(f"🤖 Response: {final_state['generated_response']}")
    print("-" * 80)
    print(f"📊 Quality Assessment:")
    print(f"   Overall Grade: {final_state['overall_grade']}")
    print(f"   Relevance: {final_state['relevance_score']:.1f}/10")
    print(f"   Accuracy: {final_state['accuracy_score']:.1f}/10")
    print(f"   Completeness: {final_state['completeness_score']:.1f}/10")
    print(f"⏱️ Processing Time: {processing_time:.2f} seconds")
    print(f"🔢 Steps Completed: {final_state['step_count']}")
    print(f"📝 Tokens Used: {final_state['tokens_used']}")
    
    if final_state['error_messages']:
        print(f"⚠️ Warnings: {final_state['error_messages']}")
    
    return final_state

print("🎉 Complete RAG Agent is ready!")
print("Use ask_rag_agent('your question') to interact with the system.")

## 8. Practical Examples and Use Cases {#examples}

### 🎯 Testing Our RAG Agent

Let's test our complete RAG system with various types of questions to see how it performs across different scenarios.

### 📚 Example Scenarios

We'll test:
1. **Direct factual questions** - Testing basic retrieval and accuracy
2. **Comparative questions** - Testing synthesis of multiple sources
3. **Conceptual questions** - Testing understanding and explanation
4. **Questions requiring inference** - Testing reasoning capabilities
5. **Questions outside our knowledge base** - Testing graceful handling of limitations

In [ ]:
# Test Example 1: Direct factual question
print("🧪 TEST 1: Direct Factual Question")
result1 = ask_rag_agent("What are the three types of AI mentioned in the documents?")
print("\n" + "="*100 + "\n")

In [ ]:
# Test Example 2: Comparative question
print("🧪 TEST 2: Comparative Question")  
result2 = ask_rag_agent("How does RAG differ from traditional machine learning approaches?")
print("\n" + "="*100 + "\n")

In [ ]:
# Test Example 3: Question outside knowledge base
print("🧪 TEST 3: Question Outside Knowledge Base")
result3 = ask_rag_agent("What is the capital of France?")
print("\n" + "="*100 + "\n")

## 10. Troubleshooting and Best Practices {#troubleshooting}

### 🔧 Common Issues and Solutions

#### Installation Problems
- **ImportError**: Update packages with `pip install --upgrade package_name`
- **Version Conflicts**: Use virtual environments to isolate dependencies
- **API Errors**: Verify OpenAI API key and check rate limits

#### Performance Issues
- **Slow Retrieval**: Reduce chunk size or number of retrieved documents
- **High Token Usage**: Optimize prompts and reduce context length
- **Memory Issues**: Process documents in smaller batches

#### Quality Issues
- **Poor Retrieval**: Improve document chunking and embedding strategy
- **Inaccurate Responses**: Enhance prompts and grading criteria
- **Inconsistent Grading**: Use more specific evaluation prompts

### 📋 Best Practices Checklist

#### Document Preparation
- ✅ Clean and preprocess documents before indexing
- ✅ Use appropriate chunk sizes (400-600 characters typically work well)
- ✅ Include metadata for better source attribution
- ✅ Regularly update the knowledge base

#### Prompt Engineering
- ✅ Use clear, specific system prompts
- ✅ Include examples in prompts when helpful
- ✅ Test prompts with various input types
- ✅ Implement prompt versioning for tracking changes

#### Quality Assurance
- ✅ Regularly evaluate response quality
- ✅ Monitor grader consistency
- ✅ Collect user feedback for improvements
- ✅ Implement A/B testing for prompt changes

#### Production Deployment
- ✅ Implement proper error handling and logging
- ✅ Set up monitoring and alerting
- ✅ Use environment variables for configuration
- ✅ Implement security measures (input sanitization, rate limiting)

In [ ]:
# Diagnostic function for troubleshooting
def run_rag_diagnostics():
    """
    Run comprehensive diagnostics for the RAG system.
    """
    print("🔍 RAG System Diagnostics")
    print("=" * 50)
    
    issues_found = []
    
    # Test 1: Check imports
    try:
        from langgraph.graph import StateGraph
        from langchain_openai import ChatOpenAI, OpenAIEmbeddings
        from langchain_community.vectorstores import FAISS
        print("✅ All required libraries imported successfully")
    except ImportError as e:
        issues_found.append(f"Import error: {e}")
        print(f"❌ Import error: {e}")
    
    # Test 2: Check API configuration
    if AZURE_OPENAI_KEY:
        print("✅ OpenAI API key configured")
    else:
        issues_found.append("OpenAI API key not found")
        print("❌ OpenAI API key not found")
    
    # Test 3: Check vector store
    if 'vector_store' in globals() and vector_store:
        print(f"✅ Vector store ready with {vector_store.index.ntotal} vectors")
    else:
        issues_found.append("Vector store not initialized")
        print("❌ Vector store not initialized")
    
    # Test 4: Check LLM connection
    try:
        test_message = [HumanMessage(content="Test connection")]
        test_response = llm.invoke(test_message)
        print("✅ LLM connection working")
    except Exception as e:
        issues_found.append(f"LLM connection error: {e}")
        print(f"❌ LLM connection error: {e}")
    
    # Test 5: Check complete workflow
    try:
        test_state = create_initial_state("What is AI?")
        print("✅ State creation working")
    except Exception as e:
        issues_found.append(f"State creation error: {e}")
        print(f"❌ State creation error: {e}")
    
    # Summary
    if not issues_found:
        print("\n🎉 All diagnostics passed! Your RAG system is ready.")
    else:
        print(f"\n⚠️ Found {len(issues_found)} issues:")
        for issue in issues_found:
            print(f"   - {issue}")
        print("\nPlease resolve these issues before proceeding.")
    
    return len(issues_found) == 0

# Run diagnostics
diagnostic_result = run_rag_diagnostics()

## 🎉 Congratulations! 

### 🏆 What You've Accomplished

You have successfully built a **complete Retrieval-Augmented Generation (RAG) agent** using LangGraph! Here's what you've mastered:

#### 🎯 Core Concepts
- ✅ **Agent AI Fundamentals**: Understanding of autonomous AI systems
- ✅ **LangGraph Architecture**: State-based workflow management
- ✅ **RAG Framework**: Retrieval, generation, and evaluation pipeline
- ✅ **State Management**: Context and memory handling in AI systems

#### 🔧 Technical Implementation
- ✅ **Retrieval Node**: Vector store creation and semantic search
- ✅ **Generation Node**: Context-aware response generation
- ✅ **Grader Node**: Automated response quality evaluation
- ✅ **Pipeline Integration**: Complete workflow orchestration

#### 🚀 Advanced Features
- ✅ **Error Handling**: Robust error management throughout the pipeline
- ✅ **Performance Monitoring**: Token tracking and processing time metrics
- ✅ **Quality Assessment**: Multi-dimensional response evaluation
- ✅ **Batch Processing**: Efficient handling of multiple queries

### 🌟 Next Steps and Applications

#### Immediate Extensions
1. **Add More Documents**: Expand your knowledge base with PDFs, web pages, or databases
2. **Custom Grading**: Develop domain-specific evaluation criteria
3. **User Interface**: Build a web app or chatbot interface
4. **Multi-modal Support**: Add support for images and other media types

#### Production Deployment
1. **Cloud Deployment**: Deploy on AWS, Azure, or Google Cloud
2. **API Development**: Create REST APIs for integration with other systems
3. **Monitoring Dashboard**: Build real-time monitoring and analytics
4. **User Management**: Add authentication and user-specific knowledge bases

#### Advanced RAG Techniques
1. **Hybrid Search**: Combine semantic and keyword search
2. **Re-ranking**: Implement sophisticated document re-ranking
3. **Multi-step Reasoning**: Add chain-of-thought reasoning capabilities
4. **Feedback Loops**: Implement learning from user interactions

### 📚 Additional Resources

- **LangGraph Documentation**: [https://python.langchain.com/docs/langgraph](https://python.langchain.com/docs/langgraph)
- **LangChain Community**: [https://github.com/langchain-ai/langchain](https://github.com/langchain-ai/langchain)
- **Vector Database Options**: Pinecone, Weaviate, Chroma, Qdrant
- **OpenAI API Documentation**: [https://platform.openai.com/docs](https://platform.openai.com/docs)

### 💡 Key Takeaways

1. **RAG is Powerful**: Combines the best of retrieval and generation
2. **Quality Matters**: Always evaluate and monitor your AI system
3. **Iterative Improvement**: Continuously refine based on feedback
4. **Production Ready**: Consider scalability, security, and monitoring from the start

**Happy building with your new RAG agent! 🤖✨**

---

### 🎮 Interactive Playground

Use the cell below to experiment with your RAG agent:

In [ ]:
# 🎮 Interactive Playground - Try your own questions!

# Modify the question below to test your RAG agent
your_question = "Explain how RAG improves upon traditional LLM responses"

print("🎯 Testing Your RAG Agent")
print("=" * 60)
print(f"Your Question: {your_question}")
print("=" * 60)

# Test your question
if 'rag_agent' in globals():
    result = ask_rag_agent(your_question)
    
    print(f"\n📋 Summary:")
    print(f"   Grade: {result['overall_grade']}")
    print(f"   Processing Time: {result['processing_time']:.2f}s")
    print(f"   Quality Scores: R={result['relevance_score']:.1f}, A={result['accuracy_score']:.1f}, C={result['completeness_score']:.1f}")
    
    # Show context used
    print(f"\n📚 Context Sources Used:")
    for i, doc in enumerate(result['retrieved_documents'][:2]):  # Show first 2 sources
        score = result['retrieval_scores'][i] if i < len(result['retrieval_scores']) else 0
        print(f"   Source {i+1} (Score: {score:.3f}): {doc.page_content[:100]}...")
        
else:
    print("❌ RAG agent not initialized. Please run the previous cells first.")

print("\n💡 Try modifying 'your_question' above and re-run this cell to test different queries!")